In [1]:
try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False
print(f"In colab: {IN_COLAB}")

In colab: True


In [2]:
if IN_COLAB:
  from pathlib import Path
  from google.colab import drive
  from datetime import datetime

  ROOT_PATH = Path('/content/drive/MyDrive/')
  OUTPUT_ROOT_PATH = ROOT_PATH / f"Colab_Data/gsm-symbolic/outputs/{datetime.now().strftime('%Y%m%d_%H%M%S')}"

  drive.mount(str(ROOT_PATH.parent))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
if IN_COLAB:
  import os

  DRIVE_PROJECT_PATH = ROOT_PATH / "Colab_Projects"
  REPO_FOLDER_NAME = 'gsm-symbolic-benchmarking'
  FULL_REPO_PATH = DRIVE_PROJECT_PATH / REPO_FOLDER_NAME

  import os
  os.chdir(FULL_REPO_PATH)
  !git pull origin main

  try:
    import gsm_benchmarker
  except ImportError:
    !pip install transformers datasets accelerate bitsandbytes numpy pandas tqdm psutil torch cloud-tpu-client anthropic openai google-genai  # other stuff probably missing too
    !pip install -e .
    os.kill(os.getpid(), 9)  # need to restart the session for import to be possible

  os.chdir("..")

From https://github.com/the-mysh/gsm-symbolic-benchmarking
 * branch            main       -> FETCH_HEAD
Already up to date.


In [4]:
import numpy as np
import logging
from huggingface_hub import login, whoami
import os

from gsm_benchmarker.dataset_wrapper import GSMSymbolicDataset
from gsm_benchmarker.benchmark_config import BenchmarkConfig
from gsm_benchmarker.benchmark import BenchmarkRunner, APIType
from gsm_benchmarker.utils.logging_setup import install_colored_logger

if IN_COLAB:
  from google.colab import userdata
  hf_api_token = userdata.get('HF_TOKEN')
  os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
  os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
else:
  hf_api_token = os.environ['HUGGINGFACEHUB_API_TOKEN']

login(hf_api_token)

whoami()['name']


'the-mysh'

In [5]:
for log_name in ('urllib3', 'fsspec', 'filelock', 'h5py', 'httpcore', 'httpx', 'google_genai', 'jax', 'root'):
    logging.getLogger(log_name).setLevel(logging.WARNING)

logger = logging.getLogger(__name__)
install_colored_logger(level=logging.DEBUG)

In [6]:
open_models = [
    'google/gemma-2-9b-it',
    'google/gemma-2-2b-it',
    'meta-llama/Meta-Llama-3-8B-Instruct',
    'microsoft/Phi-3.5-mini-instruct',
    'microsoft/Phi-3-medium-128k-instruct',
    'microsoft/Phi-3-small-128k-instruct',
    'microsoft/phi-2'
    'mistralai/Mathstral-7B-v0.1',
    'mistralai/Mistral-7B-Instruct-v0.1',
]

closed_models = [
    # ("gpt-4o", APIType.openai),  # insiffucient quota for openai
    # ("gpt-4o-mini", APIType.openai),
    # ("o1-preview", APIType.openai),
    # ("o1-mini", APIType.openai),
    ("gemini-2.0-flash", APIType.google_genai),
    ("gemini-2.5-flash", APIType.google_genai),
]

In [7]:
br = BenchmarkRunner(
    models=closed_models,  # + open_models,
    dset_variants=[GSMSymbolicDataset.Variant.main], #, GSMSymbolicDataset.Variant.p1, GSMSymbolicDataset.Variant.p2],
    storage_path=OUTPUT_ROOT_PATH,
    config=BenchmarkConfig(trust_remote_code=True)
)

gsm_benchmarker.utils.path_ops [WARNING] 2025-10-17 13:18:24: Storage root does not exist and will be created
gsm_benchmarker.utils.path_ops [INFO] 2025-10-17 13:18:24: Creating directories at /content/drive/MyDrive/Colab_Data/gsm-symbolic/outputs/20251017_131804


In [8]:
br.run(n_sets=3, n_per_set=4)

gsm_benchmarker.benchmark [INFO] 2025-10-17 13:18:24: ========== Evaluating model 1/2: gemini-2.0-flash ==========
gsm_benchmarker.model_evaluator [DEBUG] 2025-10-17 13:18:24: Initialising an API-based model
gsm_benchmarker.benchmark [INFO] 2025-10-17 13:18:24: Evaluating gemini-2.0-flash on dataset variant 1/1: Variant.main
gsm_benchmarker.dataset_wrapper [INFO] 2025-10-17 13:18:24: Loading GSM-Symbolic dataset (variant: Variant.main)...
gsm_benchmarker.dataset_wrapper [INFO] 2025-10-17 13:18:28: Loaded 5000 examples
gsm_benchmarker.dataset_wrapper [INFO] 2025-10-17 13:18:28: Creating 3 sets with 4 examples each


set:   0%|          | 0/3 [00:00<?, ?it/s]

gsm_benchmarker.utils.path_ops [INFO] 2025-10-17 13:18:28: Creating directories at /content/drive/MyDrive/Colab_Data/gsm-symbolic/outputs/20251017_131804/intermediate
gsm_benchmarker.utils.path_ops [INFO] 2025-10-17 13:18:28: Creating directories at /content/drive/MyDrive/Colab_Data/gsm-symbolic/outputs/20251017_131804/intermediate/gemini-2_0-flash
gsm_benchmarker.model_evaluator [DEBUG] 2025-10-17 13:18:28: Creating folder for results from current execution: /content/drive/MyDrive/Colab_Data/gsm-symbolic/outputs/20251017_131804/intermediate/gemini-2_0-flash/20251017_131828
gsm_benchmarker.model_evaluator [INFO] 2025-10-17 13:18:28: Intermediate results will be stored at: /content/drive/MyDrive/Colab_Data/gsm-symbolic/outputs/20251017_131804/intermediate/gemini-2_0-flash/20251017_131828


Dataset:   0%|          | 0/3 [00:00<?, ?it/s]

Example:   0%|          | 0/4 [00:00<?, ?it/s]

...

Example:   0%|          | 0/4 [00:00<?, ?it/s]

gsm_benchmarker.model_evaluator [DEBUG] 2025-10-17 13:19:32: Stored intermediate results at: /content/drive/MyDrive/Colab_Data/gsm-symbolic/outputs/20251017_131804/intermediate/gemini-2_5-flash/20251017_131851/intermediate_1.parquet


Example:   0%|          | 0/4 [00:00<?, ?it/s]

gsm_benchmarker.model_evaluator [DEBUG] 2025-10-17 13:19:48: Stored intermediate results at: /content/drive/MyDrive/Colab_Data/gsm-symbolic/outputs/20251017_131804/intermediate/gemini-2_5-flash/20251017_131851/intermediate_2.parquet
gsm_benchmarker.model_evaluator [INFO] 2025-10-17 13:19:48: Removing intermediate results
gsm_benchmarker.benchmark [DEBUG] 2025-10-17 13:19:48: Storing results
gsm_benchmarker.benchmark [DEBUG] 2025-10-17 13:19:49: Model x variant results stored to: /content/drive/MyDrive/Colab_Data/gsm-symbolic/outputs/20251017_131804/final/results/apple/GSM-Symbolic_main/gemini-2_5-flash.parquet
gsm_benchmarker.benchmark [INFO] 2025-10-17 13:19:49: Evaluation of model gemini-2.5-flash on variant Variant.main completed
gsm_benchmarker.benchmark [INFO] 2025-10-17 13:19:49: Evaluation of model gemini-2.5-flash on all dataset variants completed
gsm_benchmarker.benchmark [DEBUG] 2025-10-17 13:19:49: Deleting model from memory
gsm_benchmarker.benchmark [INFO] 2025-10-17 13:19:

{<Variant.main: 1>: {'gemini-2.0-flash':      id                                           question  true_answer  \
  0 0  81  The Charity fundraiser team hid 180 coins. The...         77.0   
    1  58  One troll can mine 5 kgs of ore per day with b...       9920.0   
    2  46  Oakwood High School is holding graduation in t...          5.0   
    3  23  Amelia is knitting winter wear for her 2 grand...        128.0   
  1 0  81  The Treasure hunt team hid 200 balls. The Lope...        166.0   
    1  58  One troll can mine 5 kgs of ore per day with w...       9800.0   
    2  46  Greenville High School is holding graduation i...          8.0   
    3  23  Aisha is knitting winter wear for her 2 grandc...        146.0   
  2 0  81  The Charity fundraiser team hid 150 coins. The...         84.0   
    1  58  One miner can mine 5 pounds of ore per day wit...       9856.0   
    2  46  Riverside Academy is holding graduation in the...          9.0   
    3  23  Chen is knitting winter we

In [9]:
br.summarise_results()

,variant,model,accuracy
0,Variant.main,gemini-2.0-flash,1.0
1,Variant.main,gemini-2.5-flash,1.0
